In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from simulator import generate_dataset
from detector import run_detection

reps = 15
windows = [1, 2, 4, 7]
recall, err = [], []
for n_days in windows:
    caught_total, fraud_total, accused = 0, 0, 0
    for rep in range(reps):
        df = pd.DataFrame(generate_dataset(n_days=n_days, fraud_rate=0.08))
        result = run_detection(df)
        fraud_total += result['is_fraudster'].sum()
        caught_total += (result['is_fraudster'] & result['flagged_driver']).sum()
        accused += (~result['is_fraudster'] & result['flagged_driver']).sum()
    r = caught_total / fraud_total
    recall.append(r)
    err.append(1.96 * (r * (1 - r) / fraud_total) ** 0.5)
    print(n_days, 'days: recall', round(r, 2), '| honest drivers accused:', accused)

plt.errorbar(windows, recall, yerr=err, marker='o', capsize=4)
plt.xlabel('Days of data observed')
plt.ylabel('Share of fraudsters caught')
plt.title('Detector power vs observation window (fraud on 8% of trips)')
plt.ylim(0, 1.05)
plt.savefig('power_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
from simulator import generate_dataset
from detector import run_detection

reps = 15
k = 5
for n_days in [1, 2, 4, 7]:
    total_hits, total_slots, total_frauds, total_drivers = 0, 0, 0, 0
    for rep in range(reps):
        df = pd.DataFrame(generate_dataset(n_days=n_days, fraud_rate=0.08))
        result = run_detection(df)
        top = result.sort_values('p_value').head(k)
        total_hits += top['is_fraudster'].sum()
        total_slots += len(top)
        total_frauds += result['is_fraudster'].sum()
        total_drivers += len(result)
    precision = total_hits / total_slots
    chance = total_frauds / total_drivers
    print(n_days, 'days: top-5 precision', round(precision, 2), '| by pure chance', round(chance, 2))

In [ ]:
import pandas as pd
from simulator import generate_dataset
from detector import run_detection

reps = 10
for varied in [False, True]:
    for n_days in [4, 7]:
        accused, hits, slots = 0, 0, 0
        for rep in range(reps):
            df = pd.DataFrame(generate_dataset(n_days=n_days, fraud_rate=0.08, varied_breaks=varied))
            result = run_detection(df)
            accused += (~result['is_fraudster'] & result['flagged_driver']).sum()
            top = result.sort_values('p_value').head(5)
            hits += top['is_fraudster'].sum()
            slots += len(top)
        label = 'varied breaks' if varied else 'identical breaks'
        print(label, n_days, 'days: honest accused', accused, '| top-5 precision', round(hits / slots, 2))

In [ ]:
import pandas as pd
from simulator import generate_dataset
from detector import run_detection

reps = 20
for varied in [False, True]:
    accused, hits, slots = 0, 0, 0
    for rep in range(reps):
        df = pd.DataFrame(generate_dataset(n_days=7, fraud_rate=0.08, varied_breaks=varied))
        result = run_detection(df)
        accused += (~result['is_fraudster'] & result['flagged_driver']).sum()
        top = result.sort_values('p_value').head(5)
        hits += top['is_fraudster'].sum()
        slots += len(top)
    label = 'skewed breaks' if varied else 'identical breaks'
    print(label, '| honest accused', accused, 'over', reps, 'runs | top-5 precision', round(hits / slots, 2))

In [ ]:
import pandas as pd
from simulator import generate_dataset
from detector import run_detection

reps = 15
tests = {'pooled baseline': ('flagged_driver', 'p_value'),
         'own baseline': ('flagged_within', 'p_value_within')}
stats = {name: {'accused': 0, 'caught': 0, 'hits': 0, 'slots': 0} for name in tests}
n_fraud = 0

for rep in range(reps):
    df = pd.DataFrame(generate_dataset(n_days=7, fraud_rate=0.08, varied_breaks=True))
    result = run_detection(df)
    n_fraud += result['is_fraudster'].sum()
    for name, (flag_col, p_col) in tests.items():
        s = stats[name]
        s['accused'] += (~result['is_fraudster'] & result[flag_col]).sum()
        s['caught'] += (result['is_fraudster'] & result[flag_col]).sum()
        top = result.sort_values(p_col).head(5)
        s['hits'] += top['is_fraudster'].sum()
        s['slots'] += len(top)

for name, s in stats.items():
    print(name, '| honest accused', s['accused'], 'over', reps, 'runs | recall',
          round(s['caught'] / n_fraud, 2), '| top-5 precision', round(s['hits'] / s['slots'], 2))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from simulator import generate_dataset
from detector import run_detection

reps = 10
methods = {'pooled baseline': 'flagged_driver', 'own baseline': 'flagged_within'}

def experiment(n_days, varied):
    out = {m: {'accused': 0, 'caught': 0} for m in methods}
    n_fraud = 0
    for rep in range(reps):
        df = pd.DataFrame(generate_dataset(n_days=n_days, fraud_rate=0.08, varied_breaks=varied))
        result = run_detection(df)
        n_fraud += result['is_fraudster'].sum()
        for m, col in methods.items():
            out[m]['accused'] += (~result['is_fraudster'] & result[col]).sum()
            out[m]['caught'] += (result['is_fraudster'] & result[col]).sum()
    return out, n_fraud

windows = [1, 2, 4, 7]
power = {d: experiment(d, True) for d in windows}
control, control_fraud = experiment(7, False)

plt.figure()
for m in methods:
    rec, err = [], []
    for d in windows:
        out, nf = power[d]
        r = out[m]['caught'] / nf
        rec.append(r)
        err.append(1.96 * (r * (1 - r) / nf) ** 0.5)
    plt.errorbar(windows, rec, yerr=err, marker='o', capsize=4, label=m)
plt.xlabel('Days of data observed')
plt.ylabel('Share of fraudsters flagged')
plt.title('Recall vs observation window (fraud on 8% of trips)')
plt.ylim(0, 1.05)
plt.legend()
plt.savefig('power_curve.png', dpi=150, bbox_inches='tight')
plt.show()

x = [0, 1]
width = 0.35
pooled = [control['pooled baseline']['accused'] / reps, power[7][0]['pooled baseline']['accused'] / reps]
own = [control['own baseline']['accused'] / reps, power[7][0]['own baseline']['accused'] / reps]
plt.figure()
plt.bar([i - width / 2 for i in x], pooled, width, label='pooled baseline')
plt.bar([i + width / 2 for i in x], own, width, label='own baseline')
plt.axhline(0.05, linestyle='--', color='gray', label='promised rate (0.05 per run)')
plt.xticks(x, ['identical breaks', 'skewed breaks'])
plt.ylabel('Honest drivers wrongly accused per run')
plt.title('False accusations, 7 days of data')
plt.legend()
plt.savefig('false_accusations.png', dpi=150, bbox_inches='tight')
plt.show()

print('pooled', pooled, '| own', own)